# Análise exploratória: JSON do INCT SiMAI

## 1. Carregamento dos dados

O arquivo `dados_inct.json` contém a simulação do INCT SiMAI: para cada nível do Guaíba (de 3,0 m a 10,0 m, em passos de 0,1 m) e para cada um dos 94 bairros de Porto Alegre, há 5 campos (`total_buildings`, `affected_buildings`, `percentage`, `population` e `affected_population`).

O JSON é um dicionário aninhado (nível → bairro → campos), formato pouco prático para análise. Por isso, o código abaixo o transforma em uma tabela no **formato longo**, com uma linha para cada combinação de nível e bairro. Duas colunas são criadas a partir das chaves do JSON: `nivel_m` (convertida de texto, como `"5_3m"`, para número, como `5.3`) e `bairro`.

**Resultado esperado:** 71 níveis × 94 bairros = 6.674 linhas e 7 colunas (2 criadas + 5 originais).

In [2]:
import json
import pandas as pd
import plotly.express as px

with open("../dados/originais/dados_inct.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])

df.shape

(6674, 7)

## Valores inconsistentes em `affected_buildings`

O número de construções afetadas deveria nunca diminuir conforme o Guaíba sobe. Foram encontradas **12 quedas**, em 5 bairros:

- **Zeros isolados** em Centro Histórico (9,7 m), Rubem Berta (8,8 m) e Lageado (8,1 m): o valor cai a 0 e volta ao normal no nível seguinte, o que parece falha da simulação ou do processamento.
- **Espírito Santo** apresenta um pico isolado (2.101 construções a 4,7 m) e uma queda de 633 para 25 em 3,5 m.
- **Arquipélago** tem oscilações pequenas (4 a 11 construções) em torno de 4.680, provavelmente ruído, já que o bairro está quase totalmente atingido desde 3,0 m.

Observação: o número de quedas não é igual ao de valores suspeitos, pois o Centro Histórico também tem um segundo zero (9,8 m) que não aparece como queda.

**Próximo passo (Fase 2):** definir como tratar esses casos, por exemplo interpolando entre os níveis vizinhos, e registrar a decisão no relatório de limpeza.

In [3]:
# Ordena por bairro e nível do rio, pra comparar cada nível com o anterior
df_ord = df.sort_values(["bairro", "nivel_m"]).copy()

# Diferença de prédios afetados entre um nível e o anterior, dentro de cada bairro
df_ord["variacao"] = df_ord.groupby("bairro")["affected_buildings"].diff()

# Fica só com os casos em que o número CAIU (deveria só crescer)
quedas = df_ord[df_ord["variacao"] < 0]

print("Quedas encontradas:", len(quedas))
quedas[["bairro", "nivel_m", "affected_buildings", "variacao"]].sort_values("variacao")

Quedas encontradas: 12


,bairro,nivel_m,affected_buildings,variacao
1717,Espírito Santo,4.8,47,-2054.0
6315,Centro Histórico,9.7,0,-2050.0
495,Espírito Santo,3.5,25,-608.0
5522,Rubem Berta,8.8,0,-252.0
4842,Lageado,8.1,0,-108.0
1037,Arquipélago,4.1,4674,-11.0
2259,Arquipélago,5.4,4680,-8.0
97,Arquipélago,3.1,4680,-5.0
567,Arquipélago,3.6,4680,-5.0
1507,Arquipélago,4.6,4680,-5.0


In [13]:
import numpy as np

def corrigir_bairro(g):
    g = g.sort_values("nivel_m").copy()
    nivel = g["nivel_m"].to_numpy(copy=True)
    v = g["affected_buildings"].to_numpy(dtype=float, copy=True)
    tol = 0.01 * g["total_buildings"].max()   # tolerância: 1% do total de prédios
    ruim = np.zeros(len(v), dtype=bool)

    # 1) Zeros depois de valores positivos e picos
    for i in range(1, len(v)):
        zero = v[i] == 0 and v[:i].max() > 0
        pico = (i < len(v) - 1 and v[i - 1] > 0
                and v[i] > 1.5 * v[i - 1] and v[i] > v[i + 1])
        if zero or pico:
            ruim[i] = True

    # 2) Valor maior que algum valor posterior (além da tolerância)
    minimo = np.inf
    for i in range(len(v) - 1, -1, -1):
        if ruim[i]:
            continue
        if v[i] > minimo + tol:
            ruim[i] = True
        else:
            minimo = min(minimo, v[i])

    if ruim.any():
        v[ruim] = np.round(np.interp(nivel[ruim], nivel[~ruim], v[~ruim]))
    g["affected_buildings_original"] = g["affected_buildings"]
    g["affected_buildings"] = v.astype(int)
    g["corrigido"] = ruim
    return g

df_limpo = pd.concat([corrigir_bairro(g) for _, g in df.groupby("bairro")], ignore_index=True)

df_limpo["percentage"] = df_limpo["percentage"].astype(float)
df_limpo["affected_population"] = df_limpo["affected_population"].astype(float)

c = df_limpo["corrigido"]
df_limpo.loc[c, "percentage"] = (df_limpo.loc[c, "affected_buildings"] / df_limpo.loc[c, "total_buildings"] * 100).round()
df_limpo.loc[c, "affected_population"] = df_limpo.loc[c, "population"] * df_limpo.loc[c, "percentage"] / 100

print("Linhas corrigidas:", c.sum())
print(df_limpo[c].groupby("bairro").size())
df_limpo[c][["bairro", "nivel_m", "affected_buildings_original", "affected_buildings"]]

Linhas corrigidas: 10
bairro
Centro Histórico    2
Espírito Santo      6
Lageado             1
Rubem Berta         1
dtype: int64


,bairro,nivel_m,affected_buildings_original,affected_buildings
1274,Centro Histórico,9.7,0,2077
1275,Centro Histórico,9.8,0,2103
1775,Espírito Santo,3.0,637,25
1776,Espírito Santo,3.1,637,25
1777,Espírito Santo,3.2,637,25
1778,Espírito Santo,3.3,637,25
1779,Espírito Santo,3.4,633,25
1792,Espírito Santo,4.7,2101,46
3459,Lageado,8.1,0,109
5028,Rubem Berta,8.8,0,269


In [14]:
df_ord = df_limpo.sort_values(["bairro", "nivel_m"]).copy()
df_ord["variacao"] = df_ord.groupby("bairro")["affected_buildings"].diff()
quedas = df_ord[df_ord["variacao"] < 0]

print("Quedas restantes:", len(quedas))
print(quedas.groupby("bairro").size())
print("Maior queda restante (prédios):", quedas["variacao"].min())

Quedas restantes: 6
bairro
Arquipélago    6
dtype: int64
Maior queda restante (prédios): -11.0


In [15]:
import os
os.makedirs("../dados/tratados", exist_ok=True)
df_limpo.to_json("../dados/tratados/dados_inct_limpo.json", orient="records", force_ascii=False, indent=2)
print("Salvo!", df_limpo.shape)

Salvo! (6674, 9)
